# Docker Basics for AI Apps

**Phase 00** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-00/08-docker-basics.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You build an AI summarizer that works perfectly on your laptop. You send it to a colleague. It fails. Different Python version. Different `anthropic` package version. Different OS. The model config you set as an environment variable on your machine never made it to theirs.

This happens constantly in AI work because AI apps have more external dependencies than typical software: specific SDK versions, API keys, model configuration, sometimes GPU drivers. The "works on my machine" failure mode kills demos, wastes review cycles, and makes production deployments unpredictable.

Docker solves this by packaging your code, its exact dependencies, and its runtime configuration into a single unit tha...

## The Concept

### Host vs. Container: What Lives Where

```
┌──────────────────────────────────────────────────────────────┐
│  HOST MACHINE                                                │
│                                                              │
│  ┌─────────────────────┐   ┌──────────────────────────────┐ │
│  │  Your filesystem    │   │  Environment variables       │ │
│  │  ~/projects/app/    │   │  ANTHROPIC_API_KEY=sk-...    │ │
│  │  ├── code/          │   │  HOME=/Users/you             │ │
│  │  │   └── main.py    │   └──────────────┬───────────────┘ │
│  │  ├── Dockerfile     │             ...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 08 - Docker Basics for AI Apps
Minimal Anthropic app that summarizes a hardcoded passage.
Run inside a container: docker run -e ANTHROPIC_API_KEY=$ANTHROPIC_API_KEY ai-summarizer
"""

import anthropic
import os
import sys

def get_client() -> anthropic.Anthropic:
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        print("Error: ANTHROPIC_API_KEY environment variable not set.", file=sys.stderr)
        print("Run with: docker run -e ANTHROPIC_API_KEY=$ANTHROPIC_API_KEY ai-summarizer", file=sys.stderr)
        sys.exit(1)
    return anthropic.Anthropic(api_key=key)


TEXT = """
The transformer architecture, introduced in 2017, replaced recurrence with
self-attention. This allowed training to be fully parallelized across tokens,
which unlocked training on much larger datasets with more parameters. By 2020,
scaling these architectures produced models that generalized across tasks
without task-specific fine-tuning.
"""


def summarize(client: anthropic.Anthropic, text: str) -> str:
    message = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=256,
        messages=[
            {
                "role": "user",
                "content": f"Summarize the following in one sentence:\n\n{text.strip()}"
            }
        ]
    )
    return message.content[0].text


def main() -> None:
    client = get_client()

    print("Input:")
    print(TEXT.strip())
    print()

    print("Calling claude-3-5-haiku-20241022...")
    summary = summarize(client, TEXT)

    print("\nSummary:")
    print(summary)
    print()
    print("Done. Container exiting with code 0.")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. What is the security problem with this approach?**

- A. The API key will be ignored because containers don't support ENV variables.
- B. The API key is embedded in the image layer history and is visible to anyone who pulls the image.
- C. Docker Hub automatically strips ENV variables from public images.
- D. The key will work but docker run will override it with an empty value.

**2. What will Docker do during the rebuild?**

- A. Reuse all cached layers because the base image hasn't changed.
- B. Reinstall all packages because COPY . . (which includes requirements.txt) comes before pip install.
- C. Only rebuild the CMD layer because CMD is always rebuilt.
- D. Skip the rebuild entirely since main.py is a small change.

**3. What is the most likely cause?**

- A. The Dockerfile is missing a COPY instruction for the .env file.
- B. The -e flag was not included in the docker run command.
- C. The Python code uses os.getenv() instead of os.environ[].
- D. The ANTHROPIC_API_KEY needs to be defined in requirements.txt.

**4. When is the slim variant the right choice for an AI API app?**

- A. Only when the app runs on machines with less than 4 GB of RAM.
- B. When the app uses only pure-Python packages and does not compile C extensions during pip install.
- C. When the app needs to import numpy or pandas.
- D. Slim images are never appropriate for production; always use the full image.

**5. Which problem does Docker solve that a Python virtual environment does NOT?**

- A. Docker ensures the same model weights are used on both machines.
- B. Docker isolates the entire OS, Python version, and system libraries, not just Python packages.
- C. Docker prevents API rate limits from affecting one machine more than another.
- D. Docker compiles Python to native code so it runs faster on any machine.

**6. Why is the second build so fast?**

- A. Docker pre-downloads base images in the background.
- B. Docker's layer cache reused every layer because no files changed.
- C. Docker skipped the CMD instruction on repeated builds.
- D. pip install is skipped on second run because packages are already in the container.

**7. Which command shows the output from a running container?**

- A. `docker inspect ai-service`
- B. `docker logs <container_id>` or `docker logs $(docker ps -lq)`
- C. `docker exec ai-service print`
- D. `docker attach --stdout ai-service`

**8. What is the correct sequence to transfer the image?**

- A. Copy the Dockerfile and run `docker build` on the client machine.
- B. Run `docker save ai-app | gzip > ai-app.tar.gz`, transfer the file, then `docker load < ai-app.tar.gz` on the client machine.
- C. Run `docker export ai-app > ai-app.tar.gz` and extract it with tar on the client machine.
- D. Push to a private registry and give the client read credentials.

_Answers are in `checks.json` in the lesson directory._